# Train **atlas** on Google Colab

Unsupervised contrastive training (ResNet-50) on ImageNette, then the few-label evaluation and figures — end to end. You watch it **live**, and on Pro+ it **keeps running even if you close the tab**.

**1. Pick the GPU** (`Runtime → Change runtime type`). On Colab Pro+:
- **L4** — the sweet spot: full run ≈ 2.6 h, **≈ 4 compute units** total (L4 ≈ 1.5 units/hr). *Recommended.*
- **A100** — faster, but burns ~8× the units/hr. Only if you want speed over budget (then use `--batch 512 --epochs 600` in cell 4).
- **T4** (free) — works (~2 h) but slower and can be preempted mid-run.

**2.** `Runtime → Run all`. Early on, cell **2b** pops up a Google-Drive auth — **click through it once** (this is what makes the result safe when you walk away). Everything else (device, mixed precision) is automatic.

**3. Watch it live:** cell 4 streams `epN/350 loss=… 27s` — that *is* your progress bar. Loss should fall from ~6.3 toward ~1.

**4. Walk away safely:** on **Pro+ the run continues in the background (~24 h)** after you close the tab. When you come back, reopen this notebook to reconnect and see the output. The encoder checkpoints every 25 epochs and evaluates the validation embedding. Then we plot the optimum epoch. When it finishes, the last cell **copies `atlas_out.zip` to your Drive** (`MyDrive/atlas/`) *and* tries a direct download — so you get the result whether or not you were watching. Hand `atlas_out.zip` back and the rest (evaluation, figures, README numbers) runs locally, no GPU.

In [ ]:
# 1) confirm the GPU
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime→GPU')

In [ ]:
# 2) clone the repo + install the light deps (torch is already on Colab)
!git clone -q https://github.com/cleoanka/atlas.git
%cd atlas
!pip install -q numpy scipy scikit-learn matplotlib pillow

In [ ]:
# 2b) Mount Google Drive so the result is SAFE even if you close the tab.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
os.makedirs('/content/drive/MyDrive/atlas', exist_ok=True)

# Verify Drive connection by writing a test file
try:
    with open('/content/drive/MyDrive/atlas/test_connection.txt', 'w') as f:
        f.write('Drive connection successful!')
    print('✅ Drive ready and verified -> results will land in MyDrive/atlas/')
except Exception as e:
    print('❌ Drive connection failed! Please check permissions:', e)


In [ ]:
# 3) fetch ImageNette (~95 MB)
!mkdir -p data && curl -sL https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz -o data/imagenette2-160.tgz
!tar -xzf data/imagenette2-160.tgz -C data/ && echo 'ready:' && ls data/imagenette2-160

In [ ]:
# 4) TRAIN (unsupervised, ResNet-50). AMP auto-on on CUDA; checkpoints every 25 epochs to models/.
#    Validated on L4 (24 GB): batch 384 fits at ~27 s/epoch. batch 512 OOMs on L4
#    (contrastive uses 2 augmented views -> effective 2x batch in memory).
#    Budget: on L4 (~1.5 units/hr) this 350-epoch run is ~2.6 h = ~4 compute units.
#    Bigger GPU? On A100 40 GB you can go --batch 512 --epochs 600 (faster, but ~8x the units/hr).
!python src/train_contrastive_imagenette.py --backbone resnet50 --img 160 --batch 384 --epochs 350 --tau 0.15

In [ ]:
# 5) Optimum Epoch Scan
# This evaluates all the intermediate embeddings and plots the edge purity / few-label accuracy.
!python figures/make_imagenette_epoch_scan.py


### Keep it (optional)

- **Google Drive** instead of downloading: `from google.colab import drive; drive.mount('/content/drive')` then `!cp data/imagenette_emb.npz models/imagenette_encoder.pt /content/drive/MyDrive/`.
- **Push back to the repo:** set a token, `!git config user.email you@x` / `user.name you`, then commit the new `data/imagenette_emb.npz` + `results/*.json` + `figures/imagenette_*.png` and `git push`.
- The `.pt` encoder is what you keep to embed new images later; the `*_emb.npz` files are what the repo's evaluation/figures use.

In [ ]:
# 6) Bundle everything and save to Drive (survives a closed tab)
import os, shutil

!cd /content/atlas && zip -q -r atlas_out.zip data/imagenette_emb*.npz models/*.pt figures/imagenette_*.png 2>/dev/null

dst = '/content/drive/MyDrive/atlas/atlas_out.zip'
if os.path.isdir('/content/drive/MyDrive/atlas'):
    shutil.copy('/content/atlas/atlas_out.zip', dst)
    # Let's also copy the raw figure and models directly so they are easy to see in Drive
    !cp /content/atlas/figures/imagenette_epoch_scan.png /content/drive/MyDrive/atlas/ 2>/dev/null
    !cp /content/atlas/models/imagenette_encoder.pt /content/drive/MyDrive/atlas/ 2>/dev/null
    print('✅ SAVED TO DRIVE ->', dst, '  (grab it whenever, even if you were away)')
else:
    print('❌ Drive not mounted — could not copy files automatically.')


In [ ]:
# 7) Disconnect runtime automatically
from google.colab import runtime
print('Finished successfully! Terminating runtime to save compute units...')
runtime.unassign()
